# 🦙 Fine-Tuning TinyLlama for Text-to-SQL with LoRA

Welcome to this hands-on tutorial! By the end of this notebook you will have **fine-tuned a 1.1B parameter LLM** to generate SQL queries from plain English — all inside a free Google Colab GPU session.

---

## 🗺️ Pipeline Overview

```
[GPU Check] → [HuggingFace Login] → [Load Model + Tokenizer] → [Quantize to 4-bit]
     → [Load Dataset] → [Format for Chat] → [Tokenize + Mask Labels]
     → [LoRA Config] → [Train] → [Save] → [Test]
```

---

## 📚 What You Will Learn
- Why **LoRA** lets you fine-tune a billion-parameter model on a laptop GPU
- How **4-bit quantization** slashes memory usage with almost no accuracy loss
- How to prepare a dataset in the **chat template** format LLMs expect
- How **label masking** teaches the model to predict only the SQL output
- How to **save and reload** LoRA adapters for later use

> **Prerequisites:** Basic Python knowledge. You do NOT need to understand every line of math — the explanations will guide you through the *why* behind each step.

## 🔧 Environment Setup

### Why this step matters
Everything in this notebook runs on a **GPU** (Graphics Processing Unit). Training even a small language model on a CPU would take hours; on a GPU it takes minutes.

The first thing we do is confirm that PyTorch can see a CUDA-capable GPU.

- `torch.cuda.is_available()` returns `True` if a GPU is ready.
- If you see `False`, go to **Runtime → Change runtime type → T4 GPU** in Colab and re-run.

> ⚠️ **Common Pitfall:** Skipping this check and only finding out you're on CPU after 30 minutes of "training" is a painful experience!

In [ ]:
import torch
torch.cuda.is_available()

True

## 🔑 HuggingFace Login

### What is HuggingFace?
[HuggingFace](https://huggingface.co) is the go-to platform for open-source AI models and datasets — think of it as *GitHub for AI*.

### Why do we need to log in?
- Some models (like Llama variants) are **gated** and require accepting a licence agreement before download.
- Logging in also lets you **push your fine-tuned model** back to HuggingFace Hub later.

### How it works here
1. Install the `huggingface_hub` library.
2. Read your secret API token from Colab's **Secrets** panel (the 🔑 icon on the left sidebar).
3. Call `login()` to authenticate — this stores the token for the session.
4. `whoami()` confirms the login worked by printing your account info.

> 💡 **Key Insight:** Storing your token in Colab Secrets (not hardcoded in the notebook) keeps it private even when you share the notebook.

# HuggingFace Login

In [ ]:
!pip install huggingface_hub

In [1]:
from huggingface_hub import login

In [2]:
from google.colab import userdata
huggingface_api = userdata.get('huggingface_api')
login(huggingface_api)

In [ ]:
from huggingface_hub import whoami
whoami()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'type': 'user',
 'id': '68e0e65b71512dfe66eda6bf',
 'name': 'Maddala-Hemanth',
 'fullname': 'Maddala Hemanth',
 'email': 'hemanthmaddala04@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1782864000,
 'isPro': False,
 'avatarUrl': '/avatars/29927c0c59b1b3ec740c0e8280a80142.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'Finetuning',
   'role': 'read',
   'createdAt': '2026-06-07T06:06:36.870Z'}}}

## 📦 Loading the Model & Tokenizer

### What is TinyLlama?
**TinyLlama-1.1B-Chat** is a compact 1.1 billion-parameter language model trained by the TinyLlama project. Despite its small size it follows the same **LLaMA 2** architecture as much larger models, making it a perfect learning sandbox.

| Property | Value |
|---|---|
| Parameters | 1.1 Billion |
| Architecture | LLaMA 2 |
| Context Length | 2048 tokens |
| Licence | Apache 2.0 (free to use) |

> 💡 **Scaling up:** Once comfortable, you can swap `model_id` to `"mistralai/Mistral-7B-Instruct-v0.3"` if you have access to a more powerful GPU (≥16 GB VRAM).

### What is a Tokenizer?
A **tokenizer** converts raw text into numbers (called *token IDs*) that the model can process.

For example:
```
"SELECT * FROM users" → [5097, 334, 3895, 4160]  (approximate)
```

Every model has its own vocabulary and tokenizer — you must always use **the tokenizer that matches the model**.

### The padding fix explained
```python
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
```
Training processes samples in **batches**, and all sequences in a batch must be the same length. Shorter sequences are "padded" with a special token. TinyLlama has no dedicated pad token, so we reuse the end-of-sequence (`eos`) token — a standard workaround.

> ⚠️ **Common Pitfall:** Forgetting to set `pad_token` causes a crash during training with a `ValueError: pad_token_id is not set` message.

# loading "model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0 model from HuggingFace_Hub'

- free to use "mistralai/Mistral-7B-Instruct-v0.3" if have GPU setup

In [3]:
!pip install -U bitsandbytes>=0.46.1 transformers

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [4]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [5]:
# Fix padding (important)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## 🧠 Understanding Quantization

### 🧠 Key Concept: Why 4-bit Quantization?

A full-precision (float32) model stores each parameter as a 32-bit number. TinyLlama at 1.1B parameters would require:

```
1.1B × 4 bytes = ~4.4 GB VRAM  (just to load, before training!)
```

**4-bit quantization** compresses each weight from 32 bits down to 4 bits — an **8× reduction** in memory:

```
1.1B × 0.5 bytes ≈ ~0.55 GB VRAM
```

This makes it possible to fine-tune on a free Colab T4 GPU (16 GB VRAM).

### How does it work?
The `BitsAndBytesConfig` object tells the loader how to quantize:

| Parameter | Value | Meaning |
|---|---|---|
| `load_in_4bit` | `True` | Compress weights to 4-bit integers |
| `bnb_4bit_compute_dtype` | `float16` | Use 16-bit math during actual computation |
| `bnb_4bit_use_double_quant` | `True` | Apply a second quantization to the scale factors — saves a little more memory |
| `bnb_4bit_quant_type` | `"nf4"` | NormalFloat4 — the best 4-bit format for normally-distributed weights |

> 💡 **Key Insight:** We *load* in 4-bit but *compute* in float16. This gives memory savings while keeping numerical stability during forward/backward passes.

> ⚠️ **Common Pitfall:** 4-bit quantization requires a CUDA GPU. Running this on a CPU-only machine will raise a `ValueError` from bitsandbytes.

# Quantization of model

In [6]:
# 4-bit quantization (VERY IMPORTANT)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16   # 🔥 ADD THIS
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## 🧪 Testing the Model *Before* Fine-Tuning

### Why test before training?
This gives us a **baseline** — we can see how the raw, un-finetuned TinyLlama responds to SQL prompts. After training, we'll compare outputs to measure how much the model improved.

### GPU memory check
We first print how much GPU VRAM the model is consuming. This is useful for:
- Confirming quantization worked (number should be small, ~1–2 GB)
- Watching out for OOM (Out-of-Memory) errors later

### Two-stage prompt experiment
1. **Simple prompt** — just asking directly without structure → expect poor/hallucinated output
2. **Structured prompt** — providing schema and a clear instruction → slightly better, but still unreliable

> 💡 **Key Insight:** This contrast shows exactly *why* fine-tuning is necessary. A general-purpose model doesn't reliably generate valid SQL without task-specific training.

> ⚠️ **Common Pitfall:** The base model may generate extra text, explanations, or completely wrong SQL. This is expected and normal — it hasn't been taught this task yet.

# Testing the Model

In [8]:
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU Memory Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU Memory Allocated: 0.73 GB
GPU Memory Reserved: 0.76 GB


In [9]:
## without finetuning
prompt = "Generate SQL: Get all users from users table"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Generate SQL: Get all users from users table
```
SELECT * FROM users;
```

2. Update a user:
```
UPDATE users SET name = 'John Doe' WHERE id = 1;
```

3. Delete a user:
```
DELETE FROM users WHERE id = 1;
```

4. Insert a new user:
```
INSERT INTO users (name, email, password) VALUES (?, ?, ?);
```

5. Update a user


In [10]:
prompt = """
You are an SQL generator.

Schema:
users(id, name, signup_date)

Question:
Get all users who signed up after 2023

SQL:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are an SQL generator.

Schema:
users(id, name, signup_date)

Question:
Get all users who signed up after 2023

SQL:
SELECT * FROM users WHERE signup_date >= DATEADD(year, -1, GETDATE());

Output:
| id | name | signup_date |
| --- | --- | --- |
| 1  | John | 2022-01-01 |
| 2  | Jane | 2023-01-01 |

Explanation:

1. The `DATEADD()` function


- models like Mistral Instruct are trained in a chat-style format.
- so converting dataset to
  - User: question
  - Assistant: answer

## 📊 Preparing the Dataset

### The dataset: `b-mc2/sql-create-context`
This is a popular open-source dataset on HuggingFace designed specifically for **Text-to-SQL** tasks. Each sample contains three fields:

| Field | Description | Example |
|---|---|---|
| `context` | SQL schema (CREATE TABLE statement) | `CREATE TABLE users (id INT, name TEXT, ...)` |
| `question` | Plain English question | `"Get all users who signed up after 2023"` |
| `answer` | Correct SQL query | `SELECT * FROM users WHERE signup_date > '2023-01-01'` |

### Why does dataset quality matter?
> 💡 **Key Insight:** Dataset quality matters more than dataset size. 1,000 clean, well-formatted examples will teach the model more than 10,000 noisy ones. Always inspect your data before training!

### What we do here
1. Install the `datasets` library (HuggingFace's data-loading toolkit)
2. Load the dataset — it downloads to a local cache automatically
3. Inspect the structure, a sample, and the field names

> ⚠️ **Common Pitfall:** Always check your dataset with `print(dataset)` and inspect a sample before starting training. Corrupted or wrongly-formatted data is a very common source of silent bugs.

# Loading the dataset

In [11]:
!pip install datasets

In [12]:
from datasets import load_dataset

dataset = load_dataset("b-mc2/sql-create-context")

In [13]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'context'],
        num_rows: 78577
    })
})


In [14]:
sample = dataset["train"][0]
print(sample)

{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}


In [15]:
schema_example = sample['context']
print(schema_example)

CREATE TABLE head (age INTEGER)


In [16]:
print(sample.keys())

dict_keys(['answer', 'question', 'context'])


## 🔄 Formatting Data for the LLM (Chat Template)

### 🧠 Key Concept: Why Chat Templates?

Modern instruction-tuned LLMs (like TinyLlama-Chat) are **not** trained on plain text. They are trained on structured **conversations** with roles:

```
[system]     → tells the model who it is
[user]       → the human's input (schema + question)
[assistant]  → the model's response (the SQL we want)
```

Providing data in exactly this format during fine-tuning is critical. If the format doesn't match what the model saw during pre-training, it will be confused and learn the wrong patterns.

### What `format_example` does
The function takes each raw dataset row and converts it into a `messages` list that mirrors a real conversation:
- **system** role: sets context (`"You are a SQL expert..."`)
- **user** role: provides the schema + question
- **assistant** role: provides the correct SQL answer

This "messages" structure is exactly what `tokenizer.apply_chat_template()` expects later.

> 💡 **Key Insight:** Models learn patterns, not true understanding. By consistently formatting data as schema → question → SQL, we teach the model the *pattern* of SQL generation. The quality of this formatting directly affects output quality.

# Dataset convertion to model style

In [17]:
def format_example(example):
    return {
        "messages": [
            {
                "role": "system",
                "content": f"""You are an SQL generator.
Return ONLY SQL query.

SCHEMA:
{example['context']}
"""
            },
            {
                "role": "user",
                "content": example["question"]
            },
            {
                "role": "assistant",
                "content": example["answer"]
            }
        ]
    }

In [18]:
formatted_dataset = dataset["train"].map(format_example)

In [19]:
print(formatted_dataset)

Dataset({
    features: ['answer', 'question', 'context', 'messages'],
    num_rows: 78577
})


In [20]:
sample = formatted_dataset[0]
print(sample)

{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)', 'messages': [{'content': 'You are an SQL generator.\nReturn ONLY SQL query.\n\nSCHEMA:\nCREATE TABLE head (age INTEGER)\n', 'role': 'system'}, {'content': 'How many heads of the departments are older than 56 ?', 'role': 'user'}, {'content': 'SELECT COUNT(*) FROM head WHERE age > 56', 'role': 'assistant'}]}


In [21]:
print(sample["messages"])

[{'content': 'You are an SQL generator.\nReturn ONLY SQL query.\n\nSCHEMA:\nCREATE TABLE head (age INTEGER)\n', 'role': 'system'}, {'content': 'How many heads of the departments are older than 56 ?', 'role': 'user'}, {'content': 'SELECT COUNT(*) FROM head WHERE age > 56', 'role': 'assistant'}]


## ⚙️ LoRA Configuration

### 🧠 Key Concept: What is LoRA?

**LoRA (Low-Rank Adaptation)** is the technique that makes fine-tuning huge models on consumer hardware possible.

#### The core idea
Instead of updating ALL 1.1 billion parameters (which requires massive GPU memory), LoRA **freezes** the original model and adds tiny **adapter layers** alongside certain weight matrices.

```
Original weight matrix W  (frozen, not updated)
       +
Small adapter: W_A × W_B  (trainable, very small)
```

Where `W_A` and `W_B` are thin matrices whose rank is controlled by `r`. If `r=16`, the total new trainable parameters are just a fraction of the original:

```
TinyLlama full params:  ~1.1 Billion
LoRA trainable params:  ~2–5 Million  (< 0.5%!)
```

#### LoRA hyperparameters explained

| Parameter | Value | Meaning |
|---|---|---|
| `r` | 16 | Rank of the adapter matrices — higher = more capacity but more memory |
| `lora_alpha` | 32 | Scaling factor; effective learning rate for LoRA weights is `alpha/r` |
| `target_modules` | `["q_proj", "v_proj"]` | Which attention layers to adapt (query and value projections) |
| `lora_dropout` | 0.05 | Dropout for regularisation — helps prevent overfitting |
| `bias` | `"none"` | Don't adapt bias terms — keeps adapter minimal |
| `task_type` | `"CAUSAL_LM"` | We are doing causal language modelling (next-token prediction) |

### `prepare_model_for_kbit_training`
This HuggingFace PEFT utility does two things:
1. Casts certain layer norms to float32 for numerical stability
2. Enables gradient checkpointing (trades compute for memory — lets you train larger batches)

### `gradient_checkpointing_enable` + `use_cache = False`
Gradient checkpointing recomputes intermediate activations during the backward pass instead of storing them — this halves memory usage at a modest speed cost. `use_cache` must be `False` during training (it's only needed during inference).

> ⚠️ **Common Pitfall:** Forgetting to call `model.config.use_cache = False` when using gradient checkpointing will trigger a warning and may cause incorrect gradient computation.

# LORA config

In [14]:
!pip install peft trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 24.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [22]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [23]:
from peft import LoraConfig, get_peft_model

In [24]:
model.gradient_checkpointing_enable()
model.config.use_cache = False
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    #target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] = takes high memory, can cause memory usage limit -> use if GPU setup is there
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [25]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


## 🔢 Tokenization & Label Masking

### 🧠 Key Concept: Why Mask Labels?

During training, the model learns to predict the **next token** given all previous tokens. But we don't want it to waste learning capacity trying to predict the system prompt or the user's question — we already know those at inference time.

We only want the model to learn to predict the **SQL answer** (the assistant's turn).

#### How label masking works

```
Full sequence:  [system prompt tokens] [user tokens] [SQL answer tokens]
Labels:         [-100, -100, -100, ...]  [-100, -100, ...]  [actual token IDs]
```

PyTorch ignores positions where the label is `-100` when computing the loss. So the model only gets gradients from the SQL answer portion.

**Without masking:** Model wastes capacity memorising prompts → worse SQL generation  
**With masking:** Model focuses entirely on learning correct SQL patterns ✅

### What the `tokenize` function does step by step
1. **`apply_chat_template`** — converts the messages list into a single formatted string with special tokens (`<|system|>`, `<|user|>`, `<|assistant|>`)
2. **Tokenize** the full text with padding/truncation to `max_length=512`
3. **Find the response start** — locate where the assistant's answer begins in the token sequence
4. **Mask** all token positions before the response with `-100`
5. Return `input_ids`, `attention_mask`, and masked `labels`

### Dataset subsetting
```python
small_dataset = tokenized_dataset.select(range(1000))
```
We use only 1,000 samples for a quick training run. The full dataset has ~78,000 samples — feel free to increase this for better results.

> ⚠️ **Common Pitfall — OOM errors:** Using too large a `max_length` or too large a batch size causes CUDA Out-of-Memory crashes. If this happens, reduce `max_length` (e.g. to 256) or reduce `per_device_train_batch_size` to 1.

> ⚠️ **Common Pitfall — Incorrect tokenization:** Always verify with `tokenizer.decode(sample['input_ids'])` that the chat template looks right before launching a full training run.

# Preparing data for training

*   x_lables => full sequence -> [system + schema + question + SQL]
*   y_lables => only sql part (but previous part is also there , but making them to -100 -> helps for next token generation type models) [-100, -100, ..., SQL tokens]



In [26]:
sample = formatted_dataset[0]
print(sample)

{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)', 'messages': [{'content': 'You are an SQL generator.\nReturn ONLY SQL query.\n\nSCHEMA:\nCREATE TABLE head (age INTEGER)\n', 'role': 'system'}, {'content': 'How many heads of the departments are older than 56 ?', 'role': 'user'}, {'content': 'SELECT COUNT(*) FROM head WHERE age > 56', 'role': 'assistant'}]}


In [27]:
output_for_supervised_finetuning = sample["messages"][-1]["content"]
print(output_for_supervised_finetuning)

SELECT COUNT(*) FROM head WHERE age > 56


In [28]:
def tokenize(example):
    # Full chat text
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length"
    )

    labels = tokenized["input_ids"].copy()

    # Get assistant text
    assistant_text = example["messages"][-1]["content"]

    # Tokenize assistant separately
    assistant_tokens = tokenizer(
        assistant_text,
        add_special_tokens=False
    )["input_ids"]

    # Find assistant start index inside input_ids
    input_ids = tokenized["input_ids"]

    start_idx = -1
    for i in range(len(input_ids) - len(assistant_tokens)):
        if input_ids[i:i+len(assistant_tokens)] == assistant_tokens:
            start_idx = i
            break

    # Mask everything BEFORE assistant
    if start_idx != -1:
        for i in range(start_idx):
            labels[i] = -100

    tokenized["labels"] = labels

    return tokenized

In [29]:
# ── 1. Tokenize the dataset ──
tokenized_dataset = formatted_dataset.map(
    tokenize,
    remove_columns=formatted_dataset.column_names
)

Map:   0%|          | 0/78577 [00:00<?, ? examples/s]

In [30]:
tokenized_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 78577
})

In [31]:
## neglect this if wanted to use whole dataset
small_dataset = tokenized_dataset.select(range(1000))

## 🚀 Training the Model (Supervised Fine-Tuning)

### The training loop — what happens inside `trainer.train()`
1. A batch of tokenized samples is loaded onto the GPU
2. The model does a **forward pass** — predicts the next token at every position
3. Loss is computed **only on the masked (SQL) positions**
4. **Backward pass** — gradients flow back through the LoRA adapter matrices
5. The optimizer updates only the LoRA weights (the base model stays frozen)
6. Repeat for all batches across all epochs

### Key `TrainingArguments` explained

| Parameter | Value | Why |
|---|---|---|
| `num_train_epochs` | 1–3 | 1 epoch is often enough; more risks overfitting on small datasets |
| `per_device_train_batch_size` | 4 | Samples processed together; reduce to 1 or 2 if you get OOM |
| `gradient_accumulation_steps` | 4 | Simulates a larger batch (effective batch = 4×4 = 16) without extra VRAM |
| `learning_rate` | 2e-4 | Standard LoRA learning rate; higher than full fine-tuning since only adapters are trained |
| `fp16` | `True` | Mixed-precision training — halves memory, speeds up compute |
| `logging_steps` | 10 | Print loss every 10 steps so you can watch progress |
| `save_strategy` | `"epoch"` | Save a checkpoint at the end of each epoch |

### `DataCollatorForLanguageModeling`
This collator dynamically pads batches to the longest sequence in each batch (instead of global max length). This is more memory-efficient than padding everything to 512 tokens.

### `SFTTrainer`
`SFTTrainer` from the `trl` library is a thin wrapper around HuggingFace's `Trainer`, optimised for **Supervised Fine-Tuning** of language models. It handles the training loop, logging, and checkpointing automatically.

> ⚠️ **Common Pitfall:** If training loss doesn't decrease at all, check that: (1) labels are correctly masked, (2) learning rate isn't too low, (3) the dataset format matches the chat template.

> 💡 **Key Insight:** The training loss should decrease steadily. If it plateaus early (e.g. stuck at 2.0+), you likely need more data or more epochs. A final loss around 0.5–1.5 is a good sign for this task.

# Training the Model (Final FineTuning)

In [32]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./TinyLlama-sql-finetuned",
    num_train_epochs=1,   # start small
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    fp16=False,
    bf16=True,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,

    report_to="none",
    dataloader_pin_memory=False,
    optim="paged_adamw_8bit"
)

In [33]:
from trl import SFTTrainer
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=small_dataset,  # or full_dataset
    args=training_args,
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [34]:
print("Starting fine-tuning...")
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting fine-tuning...


Step,Training Loss
50,1.998774


Step,Training Loss
50,1.998774
100,0.678809


TrainOutput(global_step=125, training_loss=1.1884021301269532, metrics={'train_runtime': 1516.8542, 'train_samples_per_second': 0.659, 'train_steps_per_second': 0.082, 'total_flos': 3184942645248000.0, 'train_loss': 1.1884021301269532, 'entropy': 0.5826330342888832, 'num_tokens': 512000.0, 'mean_token_accuracy': 0.8618713024258614, 'epoch': 1.0})

## 💾 Saving the Model

### What gets saved — and what doesn't
When we call `save_pretrained`, we are saving **only the LoRA adapter weights** — NOT the full 1.1B parameter base model. This is one of LoRA's biggest advantages:

| What | Size |
|---|---|
| Full TinyLlama base model | ~2.2 GB |
| LoRA adapters only | ~10–30 MB |

To use the model later, you load the base model fresh and then apply the tiny adapter on top.

### Two save locations
1. **Local Colab disk** (`./tinyllama-sql-lora`) — fast, but lost when the session ends
2. **Google Drive** (`/content/drive/MyDrive/...`) — persistent across sessions ✅

Always save to Drive if you plan to use the model after the notebook closes!

> 💡 **Key Insight:** The adapter files (`adapter_config.json` + `adapter_model.safetensors`) are all you need to recreate your fine-tuned model. You can share just these small files and anyone with the base model can reproduce your results.

# save the model

In [35]:
trainer.model.save_pretrained("tinyllama-sql-lora")
tokenizer.save_pretrained("tinyllama-sql-lora")

('tinyllama-sql-lora/tokenizer_config.json',
 'tinyllama-sql-lora/chat_template.jinja',
 'tinyllama-sql-lora/tokenizer.json')

# To goolge *drive*

In [36]:
from google.colab import drive
drive.mount('/content/drive')

trainer.model.save_pretrained("/content/drive/MyDrive/tinyllama-sql-lora")

Mounted at /content/drive


## 🧪 Testing the Fine-Tuned Model

### How inference with LoRA adapters works
Loading the fine-tuned model requires two steps:
1. Load the **base model** (quantized, just like before)
2. Wrap it with `PeftModel.from_pretrained(base_model, adapter_path)` — this overlays the LoRA adapter weights

The result behaves like a single model but is stored as base + tiny adapter.

### The generation parameters

| Parameter | Meaning |
|---|---|
| `max_new_tokens` | Maximum number of new tokens to generate — keep this short for SQL (50–100) |
| `do_sample=False` | Greedy decoding — always pick the highest-probability next token (best for deterministic SQL) |
| `temperature` | Not used with greedy, but low values (0.1) reduce randomness |

### Decoding the output
`tokenizer.decode(outputs[0], skip_special_tokens=True)` converts the token IDs back to readable text. The `skip_special_tokens=True` removes chat template tokens like `<|assistant|>` from the output.

> ⚠️ **Common Pitfall — Model generating extra text:** If the model outputs the question again before the SQL, it's because the prompt format at inference doesn't match training. Make sure you use the **same chat template** at inference as during training.

> ⚠️ **Common Pitfall — `torchao` conflict:** The `pip uninstall -y torchao` cell resolves a known compatibility issue between `torchao` and `bitsandbytes` in some Colab environments. If you skip it, you may see CUDA kernel errors when running inference.

---

## 🎉 Congratulations!

You have successfully:
- ✅ Loaded and quantized a 1.1B LLM to fit on a free GPU
- ✅ Prepared a Text-to-SQL dataset in chat template format
- ✅ Applied LoRA for memory-efficient fine-tuning
- ✅ Trained with proper label masking
- ✅ Saved and reloaded the adapter weights
- ✅ Run inference on your fine-tuned model

### 🚀 Next Steps
- **Scale up:** Use the full 78K dataset (`tokenized_dataset` instead of `small_dataset`)
- **Bigger model:** Try `mistralai/Mistral-7B-Instruct-v0.3` with a 40GB A100 GPU
- **Push to Hub:** Share your adapter with `trainer.model.push_to_hub("your-username/tinyllama-sql-lora")`
- **Evaluate properly:** Compute execution accuracy — does the SQL actually return the right rows?

# Testing

In [39]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

model = PeftModel.from_pretrained(
    base_model,
    "tinyllama-sql-lora"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [38]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [53]:
prompt = """You are an SQL generator.
Return ONLY ONE SQL query.

SCHEMA:
CREATE TABLE users(id, name, age)

Now answer:

Question: Get all users names whose age is greater than or equal to 18

SQL:"""

In [54]:
model = model.to("cuda")

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)

output = tokenizer.decode(outputs[0], skip_special_tokens=True)

sql = output.split("SQL:")[-1].strip().split("\n")[0]

print(sql)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SELECT name FROM users WHERE age >= 18
